# Gold Dimension: TfL Station

Build the historical Tube-station dimension using Slowly Changing Dimension Type 2.

The canonical TfL station NAPTAN identifier is used as the station business key.

This notebook:

1. Reads Silver StopPoint data.
2. Selects the latest reference snapshot.
3. Resolves canonical stations.
4. Creates hashes for change detection.
5. Identifies new, changed, and unchanged stations.
6. Expires changed versions.
7. Inserts new SCD versions.
8. Validates dimensional integrity.

**Source:** `workspace.urbanpulse_silver.tfl_stop_points`

**Target:** `workspace.urbanpulse_gold.dim_station`

**Business key:** `station_id`

**Grain:** One row per historical station version.

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import dependencies

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

from urbanpulse.transformations.dim_station import (
    prepare_station_source,
)

## 3. Define source and target tables

In [0]:
SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "tfl_stop_points"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "dim_station"
)

## 4. Prepare the latest station reference state

The latest Silver snapshot is reduced to one canonical record per station NAPTAN identifier.

In [0]:
silver_df = spark.table(
    SILVER_TABLE
)

source_df = prepare_station_source(
    silver_df
)

source_count = source_df.count()

print(
    f"Canonical stations: "
    f"{source_count}"
)

display(
    source_df
    .orderBy("station_name")
)

## 5. Validate station business keys

Every canonical station must have exactly one non-null `station_id`.

In [0]:
if source_count == 0:
    raise ValueError(
        "Canonical station source is empty."
    )


null_ids = (
    source_df
    .filter(
        F.col("station_id").isNull()
    )
    .count()
)


duplicate_ids_df = (
    source_df
    .groupBy("station_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)


duplicate_ids = (
    duplicate_ids_df.count()
)


if null_ids > 0:
    raise ValueError(
        "Null station IDs detected."
    )


if duplicate_ids > 0:
    display(duplicate_ids_df)

    raise ValueError(
        "Duplicate canonical station IDs "
        "detected."
    )


print(
    "Station business-key validation passed."
)

## 6. Validate station coordinates

Canonical station coordinates must be populated and fall within valid geographic ranges.

In [0]:
invalid_coordinates_df = (
    source_df
    .filter(
        F.col("latitude").isNull()
        |
        F.col("longitude").isNull()
        |
        (~F.col("latitude").between(
            -90,
            90,
        ))
        |
        (~F.col("longitude").between(
            -180,
            180,
        ))
    )
)


invalid_coordinates = (
    invalid_coordinates_df.count()
)


if invalid_coordinates > 0:
    display(
        invalid_coordinates_df
    )

    raise ValueError(
        f"{invalid_coordinates} canonical "
        "stations have invalid coordinates."
    )


print(
    "Coordinate validation passed."
)

## 7. Inspect duplicate station names

Duplicate display names are permitted because station identity is based on `station_id`, not `station_name`.

This check is informational only.

In [0]:
duplicate_names_df = (
    source_df
    .groupBy("station_name")
    .agg(
        F.countDistinct(
            "station_id"
        ).alias(
            "station_count"
        ),

        F.collect_set(
            "station_id"
        ).alias(
            "station_ids"
        ),
    )
    .filter(
        F.col("station_count") > 1
    )
    .orderBy(
        F.col(
            "station_count"
        ).desc()
    )
)

display(
    duplicate_names_df
)

## 8. Detect initial or incremental processing

In [0]:
TARGET_EXISTS = (
    spark.catalog.tableExists(
        TARGET_TABLE
    )
)

print(
    f"Target exists: "
    f"{TARGET_EXISTS}"
)

## 9. Create initial station versions

On the first execution, every canonical station becomes the first current SCD Type 2 version.

In [0]:
if not TARGET_EXISTS:

    initial_df = (
        source_df
        .withColumn(
            "effective_from",
            F.col("snapshot_at"),
        )
        .withColumn(
            "effective_to",
            F.lit(None).cast(
                "timestamp"
            ),
        )
        .withColumn(
            "is_current",
            F.lit(True),
        )
        .withColumn(
            "station_key",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("station_id"),
                    F.col("attribute_hash"),
                    F.col(
                        "effective_from"
                    ).cast("string"),
                ),
                256,
            ),
        )
        .withColumn(
            "created_at",
            F.current_timestamp(),
        )
        .withColumn(
            "updated_at",
            F.current_timestamp(),
        )
        .select(
            "station_key",
            "station_id",
            "representative_stop_point_id",
            "station_name",
            "latitude",
            "longitude",
            "stop_type",
            "modes",
            "is_active",
            "attribute_hash",
            "effective_from",
            "effective_to",
            "is_current",
            "created_at",
            "updated_at",
        )
    )

    (
        initial_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            TARGET_TABLE
        )
    )

    print(
        f"Initial dimension created: "
        f"{TARGET_TABLE}"
    )

## 10. Load current station versions

In [0]:
target_df = spark.table(
    TARGET_TABLE
)

current_df = (
    target_df
    .filter(
        F.col("is_current")
    )
)

print(
    f"Current Gold stations: "
    f"{current_df.count()}"
)

## 11. Validate source coverage

Current Gold stations must still exist in the latest reference snapshot.

Unexpected disappearance causes the pipeline to stop rather than incorrectly marking the station inactive.

In [0]:
missing_from_source_df = (
    current_df
    .select(
        "station_id",
        "station_name",
    )
    .join(
        source_df.select(
            "station_id"
        ),
        on="station_id",
        how="left_anti",
    )
)


missing_count = (
    missing_from_source_df.count()
)


if missing_count > 0:
    display(
        missing_from_source_df
    )

    raise ValueError(
        f"{missing_count} current Gold "
        "stations are missing from the "
        "latest source snapshot."
    )


print(
    "Station source coverage passed."
)

## 12. Identify new stations

In [0]:
new_stations_df = (
    source_df.alias("source")
    .join(
        current_df
        .select("station_id")
        .alias("target"),
        on="station_id",
        how="left_anti",
    )
)


new_count = (
    new_stations_df.count()
)


print(
    f"New stations: {new_count}"
)

## 13. Identify changed stations

A station is changed when its canonical business key remains the same but one or more tracked descriptive attributes have changed.

In [0]:
changed_stations_df = (
    source_df.alias("source")
    .join(
        current_df.alias("target"),
        on="station_id",
        how="inner",
    )
    .filter(
        F.col("source.attribute_hash")
        !=
        F.col("target.attribute_hash")
    )
    .select(
        "source.*"
    )
)


changed_count = (
    changed_stations_df.count()
)


print(
    f"Changed stations: "
    f"{changed_count}"
)

## 14. Identify unchanged stations

In [0]:
unchanged_stations_df = (
    source_df.alias("source")
    .join(
        current_df.alias("target"),
        on="station_id",
        how="inner",
    )
    .filter(
        F.col("source.attribute_hash")
        ==
        F.col("target.attribute_hash")
    )
    .select(
        "source.station_id",
        "source.station_name",
    )
)


unchanged_count = (
    unchanged_stations_df.count()
)


print(
    f"Unchanged stations: "
    f"{unchanged_count}"
)

## 15. Review SCD Type 2 change detection

In [0]:
print(
    "Station SCD Type 2 summary"
)

print(
    f"New:       {new_count}"
)

print(
    f"Changed:   {changed_count}"
)

print(
    f"Unchanged: {unchanged_count}"
)

## 16. Expire changed station versions

Existing current versions are closed at the timestamp of the new source snapshot.

In [0]:
if changed_count > 0:

    changed_keys_df = (
        changed_stations_df
        .select(
            "station_id",

            F.col(
                "snapshot_at"
            ).alias(
                "change_timestamp"
            ),
        )
    )

    target_delta = (
        DeltaTable.forName(
            spark,
            TARGET_TABLE,
        )
    )

    (
        target_delta.alias("target")
        .merge(
            changed_keys_df.alias(
                "source"
            ),
            """
            target.station_id
                = source.station_id
            AND target.is_current = true
            """,
        )
        .whenMatchedUpdate(
            set={
                "effective_to":
                    "source.change_timestamp",

                "is_current":
                    "false",

                "updated_at":
                    "current_timestamp()",
            }
        )
        .execute()
    )

    print(
        f"Expired {changed_count} "
        "station versions."
    )

else:
    print(
        "No station versions "
        "require expiration."
    )

## 17. Prepare new station versions

New stations and changed stations receive new current dimension versions.

In [0]:
versions_to_insert_df = (
    new_stations_df
    .unionByName(
        changed_stations_df
    )
)


versions_to_insert_df = (
    versions_to_insert_df
    .withColumn(
        "effective_from",
        F.col("snapshot_at"),
    )
    .withColumn(
        "effective_to",
        F.lit(None).cast(
            "timestamp"
        ),
    )
    .withColumn(
        "is_current",
        F.lit(True),
    )
    .withColumn(
        "station_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("station_id"),
                F.col("attribute_hash"),
                F.col(
                    "effective_from"
                ).cast("string"),
            ),
            256,
        ),
    )
    .withColumn(
        "created_at",
        F.current_timestamp(),
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
    .select(
        "station_key",
        "station_id",
        "representative_stop_point_id",
        "station_name",
        "latitude",
        "longitude",
        "stop_type",
        "modes",
        "is_active",
        "attribute_hash",
        "effective_from",
        "effective_to",
        "is_current",
        "created_at",
        "updated_at",
    )
)

## 18. Insert new station versions

Only new or changed dimension versions are appended.

In [0]:
insert_count = (
    versions_to_insert_df.count()
)


if insert_count > 0:

    (
        versions_to_insert_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            TARGET_TABLE
        )
    )

    print(
        f"Inserted {insert_count} "
        "station versions."
    )

else:
    print(
        "No new station versions "
        "to insert."
    )

## 19. Verify current station dimension

In [0]:
%sql
SELECT
    station_key,
    station_id,
    representative_stop_point_id,
    station_name,
    latitude,
    longitude,
    stop_type,
    modes,
    effective_from,
    effective_to,
    is_current
FROM workspace.urbanpulse_gold.dim_station
WHERE is_current = TRUE
ORDER BY station_name;

In [0]:
%sql
SELECT
    station_id,
    COUNT(*) AS current_versions
FROM workspace.urbanpulse_gold.dim_station
WHERE is_current = TRUE
GROUP BY station_id
HAVING COUNT(*) <> 1;

In [0]:
%sql
SELECT
    station_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_gold.dim_station
GROUP BY station_key
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT *
FROM workspace.urbanpulse_gold.dim_station
WHERE
    is_current = TRUE
    AND effective_to IS NOT NULL;

In [0]:
%sql
SELECT *
FROM workspace.urbanpulse_gold.dim_station
WHERE
    is_current = FALSE
    AND effective_to IS NULL;

## 20. Verify London geographic coverage

Tube stations should fall within a reasonable geographic bounding area around Greater London.

This check is informational because unusual source records should be investigated rather than silently removed.

In [0]:
%sql
SELECT
    station_id,
    station_name,
    latitude,
    longitude
FROM workspace.urbanpulse_gold.dim_station
WHERE
    is_current = TRUE
    AND (
        latitude NOT BETWEEN 51.20 AND 51.80
        OR longitude NOT BETWEEN -0.75 AND 0.40
    )
ORDER BY station_name;

In [0]:
%sql
SELECT
    station_name,
    COUNT(DISTINCT station_id) AS stations,
    COLLECT_SET(station_id) AS station_ids
FROM workspace.urbanpulse_gold.dim_station
WHERE is_current = TRUE
GROUP BY station_name
HAVING COUNT(DISTINCT station_id) > 1
ORDER BY stations DESC;

In [0]:
%sql
SELECT COUNT(*) AS versions
FROM workspace.urbanpulse_gold.dim_station;